# m11 lab · The EP study: intervals, pacing protocols, entrainment

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

An EP study records from catheters at a few sites and paces from them. You measure AH and HV intervals on a synthetic
His-bundle recording, map an AV nodal recovery curve with premature beats, watch Wenckebach periodicity appear with
incremental pacing, and entrain a reentrant circuit from inside and outside it.

| Part | What you do | Equation |
| --- | --- | --- |
| A | AH and HV on a His-bundle recording | eq-m11-intervals |
| B | The AV nodal recovery curve (premature beats) | eq-m11-avnode |
| C | Incremental pacing and Wenckebach periodicity | eq-m11-wenckebach |
| D | Entrainment: post-pacing interval and distance | eq-m11-ppi |
| E | Pacing rate and the post-pacing interval | eq-m11-ppi |

The AV node uses rabbit anterograde values (Reid et al. 2003) and an illustrative recovery time constant; the circuit
is a Mitchell–Schaeffer ring (m10). Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import epstudy as E
from epml.arrhythmia import traveling_pulse

## Part A · AH and HV

A catheter across the tricuspid valve records atrial (A), His (H) and ventricular (V) deflections; catheter recording of
His-bundle activity in man was described by Scherlag and colleagues (1969). The **AH** interval measures conduction
through the AV node, the **HV** interval conduction from the His bundle to the ventricles (eq-m11-intervals). Build a
recording with chosen times (illustrative), add noise and measure.

In [ ]:
t_a, t_h, t_v = 50.0, 120.0, 165.0
t, x = E.his_recording(t_a, t_h, t_v, noise=0.02, seed=3)
ta = E.local_time(t, x, (30, 80)); th = E.local_time(t, x, (100, 140)); tv = E.local_time(t, x, (145, 200))
ah, hv = E.intervals(ta, th, tv)
print(f"AH = {ah:.2f} ms (set {t_h - t_a:.0f}), HV = {hv:.2f} ms (set {t_v - t_h:.0f})")
plt.plot(t, x)
for tt, lab in ((ta, "A"), (th, "H"), (tv, "V")):
    plt.axvline(tt, color="gray", ls="--"); plt.text(tt + 2, 1.2, lab)
plt.xlim(0, 250); plt.xlabel("time (ms)"); plt.ylabel("amplitude"); plt.show()

In [ ]:
# Checks
assert abs(ah - 70.0) < 0.2 and abs(hv - 45.0) < 0.2

## Part B · The AV nodal recovery curve

Deliver a premature atrial beat after a steady train and plot nodal conduction time against the time the node has had
to recover since it last conducted (eq-m11-avnode). Rabbit preparations gave a minimum nodal conduction time of 57 ms,
a maximum of 150 ms at the nodal effective refractory period of 91 ms (Reid et al. 2003).

In [ ]:
rts = np.arange(60, 500, 5.0)
nct = E.nodal_time(rts)
erp_meas = rts[np.isfinite(nct)][0]
print(f"shortest recovery time that conducts: {erp_meas:.0f} ms; conduction time there {nct[np.isfinite(nct)][0]:.0f} ms")
print(f"conduction time after 400 ms of recovery: {float(E.nodal_time(400.0)):.1f} ms")
plt.plot(rts, nct); plt.axvline(E.AVN["erp"], color="gray", ls="--")
plt.xlabel("recovery time (ms)"); plt.ylabel("nodal conduction time (ms)"); plt.show()

In [ ]:
# Checks
assert E.AVN["erp"] <= erp_meas < E.AVN["erp"] + 5          # first grid point at or above ERP_N
assert np.all(np.diff(nct[np.isfinite(nct)]) < 0)

## Part C · Incremental pacing and Wenckebach periodicity

Pace the atrium at a fixed cycle length. Each beat's recovery time is the cycle length minus the previous beat's
nodal time, so a long nodal time leaves less recovery for the next beat. With a stable fixed point, conduction stays
1:1 down to $\text{ERP}_N + \text{NCT}_{\max}$ (eq-m11-wenckebach); below it, nodal times lengthen until a beat
blocks, and the pattern repeats.

In [ ]:
cl_w, slope = E.wenckebach_cl()
print(f"Wenckebach point ERP_N + NCT_max = {cl_w:.0f} ms; (NCT_max - NCT_min) / tau = {slope:.2f}")
ratios = {}
for cl in (300, 250, 242, 235, 220, 200, 150):
    seq = E.av_sequence(float(cl), 60)
    ratios[cl] = E.conduction_ratio(seq)
    print(f"CL {cl}: {ratios[cl][0]}:{ratios[cl][1]}, last nodal times {np.round(seq[-6:], 0)}")
seq = E.av_sequence(220.0, 30)
plt.plot(np.where(np.isfinite(seq), seq, np.nan), "o-"); plt.plot(np.where(np.isnan(seq))[0], np.zeros(np.isnan(seq).sum()), "rx")
plt.xlabel("atrial beat"); plt.ylabel("nodal conduction time (ms)"); plt.title("CL 220 ms"); plt.show()

In [ ]:
# Checks
assert ratios[242] == (1, 1) and ratios[220] == (4, 3) and ratios[150] == (2, 1)
assert slope < 1

## Part D · Entrainment and the post-pacing interval

Reentry runs on a 100-mm Mitchell–Schaeffer ring; a dead-end branch of length $d$ joins it at one point. Pace at the
end of the branch 20 ms faster than the tachycardia, stop, and measure the post-pacing interval (PPI) at the pacing
site. Outside the circuit the last paced wave must travel in, around and back out, so
$\text{PPI} - \text{TCL} \approx 2d/\text{CV}$ (eq-m11-ppi). About 20 s.

In [ ]:
pulse = traveling_pulse()
res = {}
for d in (0.0, 10.0, 20.0):
    tcl, ppi, cvb = E.entrainment(ring=100.0, d=d, offset=20.0, pulse=pulse)
    res[d] = (tcl, ppi, cvb)
    pred = 2 * d / cvb if np.isfinite(cvb) else 0.0
    print(f"d = {d:4.1f} mm: TCL {tcl:.0f} ms, PPI {ppi:.0f} ms, PPI - TCL {ppi - tcl:.0f} ms; 2d/CV = {pred:.0f} ms")

In [ ]:
# Checks
assert 0 <= res[0.0][1] - res[0.0][0] < 20
for d in (10.0, 20.0):
    tcl, ppi, cvb = res[d]
    assert abs((ppi - tcl) - 2 * d / cvb) < 15
assert res[20.0][1] - res[20.0][0] > res[10.0][1] - res[10.0][0] > res[0.0][1] - res[0.0][0]

## Part E · Pacing rate

Pacing on the circuit should give a PPI close to the cycle length, but pacing faster slows conduction in the circuit
(the waves meet less recovered tissue), and that slowing lengthens the PPI too. Stevenson and colleagues noted this
in simulations (1993). Compare pacing 20 and 60 ms faster than the tachycardia.

In [ ]:
tcl, ppi20, _ = E.entrainment(ring=100.0, d=0.0, offset=20.0, pulse=pulse)
_, ppi60, _ = E.entrainment(ring=100.0, d=0.0, offset=60.0, pulse=pulse)
print(f"on the circuit: PPI - TCL = {ppi20 - tcl:.0f} ms pacing 20 ms faster, {ppi60 - tcl:.0f} ms pacing 60 ms faster")

In [ ]:
# Checks
assert ppi60 - tcl > ppi20 - tcl + 15

## Where to go next

- In part C, set $\tau$ to 50 ms so that $(\text{NCT}_{\max} - \text{NCT}_{\min})/\tau > 1$: where does 1:1 conduction
  end now, and why?
- In part D, make the branch conduct slowly near the junction: how does the PPI change?
- m22 and m24 show how the timing of local electrograms is measured on real recordings.